# GRPO fine-tuning on top of the macro LoRA checkpoint (PlurVA zh/id/si)

Continues training from `adapters/macro_lora_pt/best` -- the simultaneous zh/id/si
macro-averaged LoRA checkpoint that scored **0.7501 macro accuracy** -- using GRPO
(Group Relative Policy Optimization) instead of supervised cross-entropy.

**Mechanism**: for each prompt, sample a *group* of G completions from the current
policy (temperature > 0), score each with a reward (did it output the correct
letter?), normalize rewards within the group (subtract group mean, divide by group
std) to get a per-sample advantage, then do a policy-gradient update that pushes up
log-probability of above-average completions and down below-average ones. No
critic/value network needed (that's what makes it "group relative" instead of PPO).

**Design choices / simplifications, stated explicitly:**
- **Reward** = 1.0 if the extracted answer letter is in the row's gold candidate set
  (reuses `resolve_gold_candidates` from `eval_baseline.py`, so Indonesian's tied-vote
  rows -- e.g. "C, D, A, A, D" -- correctly count *either* tied letter as correct,
  same logic as the SFT tie-duplication fix), 0.0 if wrong-but-parseable, and a small
  -0.2 penalty if the completion is unparseable (`extract_letter` returns `None`), to
  discourage garbled output.
- **Balanced per-language row sampling per step**, mirroring `train_macro_lora_pt.py`'s
  macro design: every step samples the same number of rows from each language.
- **Row-level train/val split is reconstructed with the same seed/logic as
  `prepare_training_data.py`** (seed=42, val_fraction=0.15, shuffled at the row level)
  so GRPO only trains on rows the SFT checkpoint also treated as "train", and can be
  validated against the same held-out rows.
- **No explicit KL-to-reference penalty.** A full GRPO/RLHF setup usually keeps a
  frozen reference copy of the model and penalizes divergence from it. Skipped here to
  avoid holding two 4B-parameter model copies in memory simultaneously; if the policy
  drifts or reward hacks (e.g. always emitting "A"), that's the first thing to add back.
- **Degenerate groups are skipped**: if every sample in a group gets the same reward,
  the group's advantage is all zeros and contributes no gradient -- this is expected,
  not a bug, and is why GRPO generally needs a policy that isn't already at 0% or 100%
  on a given prompt to learn anything from it.

Runs on CUDA, MPS, or CPU (auto-detected, CUDA strongly preferred -- GRPO requires
actual autoregressive sampling every step, which is far slower than SFT's single
teacher-forced forward pass).

## Imports

In [ ]:
import json
import random
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


Reuse the exact prompt-building / gold-resolution / letter-extraction logic from `scripts/eval_baseline.py`, so GRPO's prompts and reward signal match the SFT pipeline exactly.

In [ ]:
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = Path.cwd().parent  # fallback if launched from a subdirectory

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from eval_baseline import load_rows, resolve_gold_candidates, resolve_options, build_prompt, extract_letter

LANGS = ["zh", "id", "si"]


## Reconstruct the train/val row split

Mirrors `prepare_training_data.py`'s `build_example_groups` split exactly (same
seed, same `val_fraction`, same shuffle-then-slice-by-row logic) but returns raw
rows (with `Option_A..D` intact) instead of flattened prompt/completion examples,
since GRPO needs the options text and the full gold-candidate set per row, not a
single pre-picked completion string.

In [ ]:
SEED = 42
VAL_FRACTION = 0.15


def load_split_rows(lang):
    rows = load_rows(lang)
    rng = random.Random(SEED)
    order = list(range(len(rows)))
    rng.shuffle(order)
    n_val = max(1, int(len(rows) * VAL_FRACTION))
    val_idx = set(order[:n_val])
    train_rows = [rows[i] for i in order if i not in val_idx]
    val_rows = [rows[i] for i in order if i in val_idx]
    return train_rows, val_rows


train_rows_by_lang = {}
val_rows_by_lang = {}
for lang in LANGS:
    train_rows_by_lang[lang], val_rows_by_lang[lang] = load_split_rows(lang)
    print(f"{lang}: {len(train_rows_by_lang[lang])} train rows, {len(val_rows_by_lang[lang])} val rows")


**Important caveat**: `prepare_training_data.py` shuffles at the *row* level too
(as of the tie-duplication fix), but zh/id/si are shuffled in sequence off a single
shared `random.Random(42)` instance there, so id's list length affects the RNG state
si's shuffle sees. Reproducing that exact cross-language coupling here isn't
possible without re-running that script's full loop, so this cell instead shuffles
each language independently with a fresh `Random(SEED)`. That means this val split
won't be byte-identical to `data/val/val_<lang>.jsonl` row-for-row, but it uses the
same seed, fraction, and per-row logic, and -- more importantly -- is internally
consistent for GRPO's own train/val separation. If exact parity with the SFT split
matters, load row IDs out of `data/train/train_<lang>.jsonl` / `data/val/val_<lang>.jsonl`
directly instead and filter `load_rows(lang)` against those.

## Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
START_ADAPTER_PATH = str(REPO_ROOT / "adapters" / "macro_lora_pt" / "best")  # the 75.01% macro-acc checkpoint
OUT_ADAPTER_PATH = REPO_ROOT / "adapters" / "macro_lora_grpo"  # GRPO writes here, never overwrites the SFT checkpoint

ROWS_PER_LANG_PER_STEP = 1   # balanced per-language sampling, mirrors train_macro_lora_pt.py
GROUP_SIZE = 8               # completions sampled per prompt (the "group" in GRPO)
TEMPERATURE = 0.8
TOP_P = 0.95
MAX_NEW_TOKENS = 40
UNPARSEABLE_PENALTY = -0.2

ITERS = 200
LEARNING_RATE = 5e-6         # notably lower than SFT's 1e-4 -- RL updates are noisier
MAX_SEQ_LENGTH = 768
STEPS_PER_EVAL = 10
VAL_ROWS_PER_LANG = 10       # rows per language sampled (greedy) at each eval
STEPS_PER_SAVE = 20
SEED_RUN = 42


## Load model + tokenizer + the existing LoRA adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
dtype = torch.float16 if device == "cuda" and not torch.cuda.is_bf16_supported() else torch.bfloat16

print(f"Loading {MODEL_ID} on {device} ({dtype}) ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)
print(f"Loading trainable LoRA adapter from {START_ADAPTER_PATH} ...")
model = PeftModel.from_pretrained(base_model, START_ADAPTER_PATH, is_trainable=True)
model.to(device)
model.train()

rng = random.Random(SEED_RUN)


## Sampling + reward

For one row: build the same chat-templated prompt used everywhere else, sample
`GROUP_SIZE` completions, extract each one's predicted letter, and score it against
the row's gold candidate set.

In [ ]:
def build_chat_prompt(lang, row, options):
    prompt = build_prompt(lang, row, options)
    messages = [{"role": "user", "content": prompt}]
    try:
        return prompt, tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )
    except TypeError:
        return prompt, tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
        )


def reward_fn(pred_letter, gold_candidates):
    if pred_letter is None:
        return UNPARSEABLE_PENALTY
    return 1.0 if pred_letter in gold_candidates else 0.0


@torch.no_grad()
def sample_group(lang, row, group_size, do_sample=True):
    """Returns a list of dicts: {input_ids (prompt+completion), prompt_len, reward}."""
    options = resolve_options(lang, row)
    valid_letters = {l for l in "ABCD" if options[l]}
    gold_candidates = set(resolve_gold_candidates(lang, row["Gold_Answer"]))
    _, chat_prompt = build_chat_prompt(lang, row, options)

    inputs = tokenizer(chat_prompt, return_tensors="pt", add_special_tokens=False).to(device)
    prompt_len = inputs["input_ids"].shape[1]

    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        num_return_sequences=group_size,
    )
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
    else:
        gen_kwargs.update(do_sample=False)

    output_ids = model.generate(**inputs, **gen_kwargs)

    samples = []
    for seq in output_ids:
        completion_ids = seq[prompt_len:].tolist()
        # trim trailing pad tokens so labels/loss don't include them
        while completion_ids and completion_ids[-1] == tokenizer.pad_token_id:
            completion_ids.pop()
        response = tokenizer.decode(completion_ids, skip_special_tokens=True)
        pred = extract_letter(response, valid_letters, options)
        reward = reward_fn(pred, gold_candidates)
        samples.append({
            "input_ids": inputs["input_ids"][0].tolist() + completion_ids,
            "prompt_len": prompt_len,
            "reward": reward,
        })
    return samples


## GRPO loss for one step

Normalizes rewards within each row's group to get advantages, skips degenerate
groups (zero reward variance -> zero gradient anyway), then does a single combined
forward+backward over all surviving samples across all languages in the step
(unlike `train_macro_lora_pt.py`'s per-language split -- here the sampling step
already dominates the memory/time budget, so there's no equivalent motivation to
split the backward pass by language).

In [ ]:
def collate_weighted(samples, pad_token_id, device):
    max_len = max(len(s["input_ids"]) for s in samples)
    input_ids = torch.full((len(samples), max_len), pad_token_id, dtype=torch.long)
    labels = torch.full((len(samples), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(samples), max_len), dtype=torch.long)
    weights = torch.zeros(len(samples), dtype=torch.float)
    for i, s in enumerate(samples):
        ids = s["input_ids"]
        L = len(ids)
        input_ids[i, :L] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :L] = 1
        labels[i, s["prompt_len"]:L] = torch.tensor(ids[s["prompt_len"]:], dtype=torch.long)
        weights[i] = s["advantage"]
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
        "weights": weights.to(device),
    }


def grpo_step(batches_by_lang):
    """batches_by_lang: {lang: [row, ...]} -- ROWS_PER_LANG_PER_STEP rows per language."""
    all_samples = []
    reward_log = {}
    for lang, rows in batches_by_lang.items():
        lang_rewards = []
        for row in rows:
            group = sample_group(lang, row, GROUP_SIZE, do_sample=True)
            rewards = torch.tensor([s["reward"] for s in group])
            lang_rewards.extend(rewards.tolist())
            std, mean = rewards.std(unbiased=False), rewards.mean()
            if std < 1e-6:
                continue  # degenerate group: every sample scored the same, no signal
            advantages = (rewards - mean) / (std + 1e-6)
            for s, adv in zip(group, advantages.tolist()):
                s["advantage"] = adv
                all_samples.append(s)
        reward_log[lang] = sum(lang_rewards) / len(lang_rewards) if lang_rewards else float("nan")

    if not all_samples:
        return None, reward_log  # every group this step was degenerate

    batch = collate_weighted(all_samples, tokenizer.pad_token_id, device)
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out.logits[:, :-1, :]
    labels = batch["labels"][:, 1:]

    token_ce = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view(labels.shape)
    valid = (labels != -100)
    tok_count = valid.sum(dim=1).clamp(min=1)
    per_example_loss = (token_ce * valid).sum(dim=1) / tok_count  # mean CE per example = -mean logprob
    weighted_loss = (per_example_loss * batch["weights"]).mean()

    weighted_loss.backward()
    return weighted_loss.item(), reward_log


## Validation (greedy accuracy on held-out rows)

In [ ]:
@torch.no_grad()
def validate(n_rows_per_lang):
    model.eval()
    lang_acc = {}
    for lang in LANGS:
        rows = rng.sample(val_rows_by_lang[lang], min(n_rows_per_lang, len(val_rows_by_lang[lang])))
        correct = 0
        for row in rows:
            sample = sample_group(lang, row, group_size=1, do_sample=False)[0]
            correct += 1 if sample["reward"] == 1.0 else 0
        lang_acc[lang] = correct / len(rows)
    model.train()
    return lang_acc, sum(lang_acc.values()) / len(lang_acc)


## Training loop

In [ ]:
OUT_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

best_val_macro = -1.0
t0 = time.time()
pbar = tqdm(range(1, ITERS + 1), desc="grpo", unit="it")
for it in pbar:
    batches_by_lang = {
        lang: [rng.choice(train_rows_by_lang[lang]) for _ in range(ROWS_PER_LANG_PER_STEP)]
        for lang in LANGS
    }

    optimizer.zero_grad()
    loss_value, reward_log = grpo_step(batches_by_lang)
    if loss_value is not None:
        optimizer.step()

    reward_str = " ".join(f"r[{lang}]={reward_log[lang]:.2f}" for lang in LANGS)
    pbar.set_postfix(loss=f"{loss_value:.4f}" if loss_value is not None else "skip")
    tqdm.write(f"[iter {it}/{ITERS}] loss={loss_value if loss_value is not None else float('nan'):.4f} {reward_str}")

    if it % STEPS_PER_EVAL == 0 or it == ITERS:
        lang_acc, val_macro = validate(VAL_ROWS_PER_LANG)
        acc_str = " ".join(f"val_acc[{lang}]={lang_acc[lang]:.3f}" for lang in LANGS)
        tqdm.write(f"[iter {it}] val_macro_acc={val_macro:.4f} {acc_str}")
        if val_macro > best_val_macro:
            best_val_macro = val_macro
            model.save_pretrained(str(OUT_ADAPTER_PATH / "best"))
            tqdm.write(f"[iter {it}] New best val macro acc; saved adapter to {OUT_ADAPTER_PATH / 'best'}")

    if it % STEPS_PER_SAVE == 0 or it == ITERS:
        model.save_pretrained(str(OUT_ADAPTER_PATH))
        tqdm.write(f"[iter {it}] Saved latest adapter to {OUT_ADAPTER_PATH}")

elapsed = time.time() - t0
print(f"Done in {elapsed:.0f}s. Best val macro acc: {best_val_macro:.4f}. "
      f"Best adapter: {OUT_ADAPTER_PATH / 'best'}, latest: {OUT_ADAPTER_PATH}")
